In [0]:
select count(*) from ws_dbxproject.`01_bronze`.historical_orders;
select count(*) from ws_dbxproject.`01_bronze`.reviews;
select * from ws_dbxproject.`02_silver`.dim_customers
where customer_id='CUST-10000';

select count(*) from ws_dbxproject.`02_silver`.dim_menu_items;


select * from ws_dbxproject.`01_bronze`.orders;

-- Backfill: copy all historical orders into the bronze orders table (append, no dedup)
insert into ws_dbxproject.`01_bronze`.orders
select * from ws_dbxproject.`01_bronze`.historical_orders;

select * from ws_dbxproject.`02_silver`.fact_orders
select count(*) from ws_dbxproject.`02_silver`.fact_orders

select * from ws_dbxproject.`02_silver`.fact_order_items

select * from ws_dbxproject.`02_silver`.fact_reviews

drop table ws_dbxproject.`02_silver`.fact_orders

select * from ws_dbxproject.`02_silver`.fact_reviews


-- Dashboard Data sets
-- dim_customers_sql
select * from ws_dbxproject.`02_silver`.dim_customers
where join_date between :DATE_RANGE.min and :DATE_RANGE.max;

-- fact_orders_sql
select * from ws_dbxproject.`02_silver`.fact_orders
where order_date between :DATE_RANGE.min and :DATE_RANGE.max;

-- d_sales_summary_sql
select * from ws_dbxproject.`03_gold`.d_sales_summary
where order_date between :DATE_RANGE.min and :DATE_RANGE.max;

-- fact_order_items_sql
select item_name,
sum(quantity) as total_qty_sold
from ws_dbxproject.`02_silver`.fact_order_items
where order_date between :Date_range.min and :Date_range.max
group by item_name
order by total_qty_sold desc
limit 10

-- fact_order_items_1_sql
select * from ws_dbxproject.`02_silver`.fact_order_items
where order_date between :Date_range.min and :Date_range.max

-- linesql2
select rr.restaurant_name,
date(review_timestamp) as review_date,
count(distinct case when sentiment = 'positive' then order_id else null end) as positive_review_count,
count(distinct case when sentiment = 'neutral' then order_id else null end) as neutral_review_count,
count(distinct case when sentiment = 'negative' then order_id else null end) as negative_review_count
from ws_dbxproject.`02_silver`.fact_reviews fr 
Join ws_dbxproject.`03_gold`.d_restaurant_reviews rr 
on fr.restaurant_id = rr.restaurant_id
group by 1,2
order by 1,2

-- ratingcount
select restaurant_name,
rating_label,
rating_count
from ws_dbxproject.`03_gold`.d_restaurant_reviews
lateral view stack(
    5,
    "5 star",rating_5_count,
    "4 star",rating_4_count,
    "3 star",rating_3_count,
    "2 star",rating_2_count,
    "1 star",rating_1_count
) As rating_label, rating_count


-- issue_summary
select dr.name as restaurant_name,
count(distinct case when fr.issue_delivery = true then order_id else null end) as count_issue_delivery,
count(distinct case when fr.issue_food_quality = true then order_id else null end) as count_issue_food_quality,
count(distinct case when fr.issue_pricing = true then order_id else null end) as count_issue_pricing,
count(distinct case when fr.issue_portion_size = true then order_id else null end) as count_issue_portion_size 
from ws_dbxproject.`02_silver`.fact_reviews fr 
join ws_dbxproject.`02_silver`.dim_restaurants dr 
on fr.restaurant_id = dr.restaurant_id
group by 1

-- Recent Positive
select dr.name as restaurant_name,
review_text,review_timestamp
from ws_dbxproject.`02_silver`.fact_reviews  fr
join ws_dbxproject.`02_silver`.dim_restaurants dr
on fr.restaurant_id = dr.restaurant_id
where fr.sentiment ='positive' and fr.review_text is NOT NULL
order by review_timestamp desc

-- Recent Negative
select dr.name as restaurant_name,
review_text,review_timestamp
from ws_dbxproject.`02_silver`.fact_reviews  fr
join ws_dbxproject.`02_silver`.dim_restaurants dr
on fr.restaurant_id = dr.restaurant_id
where fr.sentiment ='negative' and fr.review_text is NOT NULL
order by review_timestamp desc





